In [0]:
# =============================================================================
# NOTEBOOK  : silver_store_inventory
# PURPOSE   : bronze.store_inventory → silver.store_inventory
# LAYER     : Silver (clean, flatten nested JSON, merge)
# FREQUENCY : Every 15 minutes (availableNow)
# TRIGGER   : availableNow
#
# TRANSFORM:
#   - last_restocked_date : String → DateType
#   - expiry_date         : String → DateType (null for non-perishables)
#   - temperature_reading : nested JSON string → flattened into 3 columns:
#                           sensor_id, temperature_celsius, humidity
#                           null for non temperature-sensitive products
#
# AUDIT COLUMNS:
#   ingested_at, processed_at, pipeline_run_id, source_system
#
# MERGE & DEDUP LOGIC:
#   Merge key : store_id + product_id (latest snapshot per store-product pair)
#   Dedup     : window on store_id + product_id ordered by ingested_at DESC
#               15-min snapshots may have multiple entries for same pair
#               keep most recently ingested snapshot
#               done BEFORE .select() so ingested_at available as tiebreaker
#
#   Case 1 : store_id + product_id NOT in silver → INSERT
#   Case 2 : store_id + product_id IN silver, any value changed → UPDATE
#            all columns updatable — this is a snapshot table
#            every new snapshot reflects current store shelf state
#   Case 3 : store_id + product_id IN silver, no change → IGNORE
#
# NOTE: temperature_reading is null for non temperature-sensitive products
#       expiry_date is null for non-perishables
#       both nulls are valid and preserved in silver
# =============================================================================

# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_STORE_INVENTORY_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, get_json_object, row_number
)
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["bronze_store_inventory"]
TARGET_TABLE = cfg["tables"]["silver_store_inventory"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_store_inventory"]
PIPELINE     = "silver_store_inventory"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────

def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze store inventory rows.

    DEDUP  : window on store_id + product_id ordered by ingested_at DESC
             done BEFORE .select() so ingested_at available as tiebreaker
             keeps most recently ingested snapshot per store-product pair

    MERGE  :
      Case 1 → store_id + product_id not in silver → INSERT
      Case 2 → pair in silver, any value changed   → UPDATE all snapshot cols
      Case 3 → pair in silver, no change           → IGNORE

    FLATTEN:
      temperature_reading JSON string extracted into 3 flat columns:
        sensor_id, temperature_celsius, humidity
      null when product is not temperature-sensitive
    """

    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_STORE_INVENTORY_SCHEMA

    if micro_batch_df.isEmpty():
        return

    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

    try:
        rows_read = micro_batch_df.count()

        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df

            # 1. Date columns: String → DateType
            .withColumn("last_restocked_date",
                to_date(col("last_restocked_date")))
            .withColumn("expiry_date",
                to_date(col("expiry_date")))

            # 2. Flatten temperature_reading nested JSON string
            #    Source: {"sensor_id":"SENSOR_27",
            #             "temperature_celsius":1.62,"humidity":61.64}
            #    null when product is not temperature-sensitive
            #    get_json_object returns null when input is null — safe
            .withColumn("sensor_id",
                get_json_object(col("temperature_reading"), "$.sensor_id"))
            .withColumn("temperature_celsius",
                get_json_object(col("temperature_reading"), "$.temperature_celsius")
                .cast(DoubleType()))
            .withColumn("humidity",
                get_json_object(col("temperature_reading"), "$.humidity")
                .cast(DoubleType()))

            # 3. Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("store_inventory_jsonl"))
            # ingested_at carried from bronze automatically via .select()
        )

        # ── DEDUP WITHIN MICRO-BATCH ──────────────────────────────────────────
        df = df.dropDuplicates(["store_id", "product_id"])

        # ── ENFORCE SILVER SCHEMA ─────────────────────────────────────────────
        # Done AFTER dedup so ingested_at was available for window ordering
        # Drops bronze-only columns:
        # (temperature_reading raw string, source_file, ingested_date, etc.)
        df = df.select([f.name for f in SILVER_STORE_INVENTORY_SCHEMA.fields])

        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # This is a snapshot table — all columns are updatable
        # Case 2: any value changed → UPDATE all snapshot columns
        # Case 1: new store-product pair → INSERT
        # Case 3: no change → no rule fires → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                """
                t.store_id   = s.store_id AND
                t.product_id = s.product_id
                """
            )
            .whenMatchedUpdate(
                condition="""
                    t.current_quantity    != s.current_quantity     OR
                    t.last_restocked_date != s.last_restocked_date  OR
                    t.shelf_location      != s.shelf_location        OR
                    t.expiry_date         != s.expiry_date           OR
                    t.sensor_id           != s.sensor_id             OR
                    t.temperature_celsius != s.temperature_celsius   OR
                    t.humidity            != s.humidity
                """,
                set={
                    "current_quantity":    "s.current_quantity",
                    "last_restocked_date": "s.last_restocked_date",
                    "shelf_location":      "s.shelf_location",
                    "expiry_date":         "s.expiry_date",
                    "sensor_id":           "s.sensor_id",
                    "temperature_celsius": "s.temperature_celsius",
                    "humidity":            "s.humidity",
                    "processed_at":        "current_timestamp()",
                    "pipeline_run_id":     f"'{run_id}'"
                    # ingested_at not updated — preserve original bronze landing time
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_inserted + rows_updated,
            extra={
                "rows_inserted": str(rows_inserted),
                "rows_updated":  str(rows_updated),
                "microbatch_id": str(microbatch_id)
            }
        )

    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise


# ── READ BRONZE AS STREAM ─────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)

# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col

# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)

# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())

# 3. Nulls in key columns
print("Null key columns:",
    spark.read.table(TARGET_TABLE)
    .filter(col("store_id").isNull() | col("product_id").isNull())
    .count()
)

# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)